In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
import calendar
from datetime import date, timedelta
from itertools import product

from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")
catalogo = dbutils.widgets.get("catalogo")

In [0]:
s_viajes     = f"{catalogo}.silver.viajes"
s_estaciones = f"{catalogo}.silver.estaciones"
s_clima     = f"{catalogo}.silver.clima_hora"

In [0]:
G_DIM_FECHA    = f"{catalogo}.golden.dim_fecha"
G_DIM_ESTACION = f"{catalogo}.golden.dim_estacion"
G_DIM_USUARIO  = f"{catalogo}.golden.dim_usuario"
G_DIM_CLIMA    = f"{catalogo}.golden.dim_clima"
G_FACT_CLIMA   = f"{catalogo}.golden.fact_clima_hora"
G_FACT_VIAJES  = f"{catalogo}.golden.fact_viajes"

In [0]:
def guardar(df, tabla):
    (
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(tabla)
    )
    print(f"{tabla}: {spark.table(tabla).count():,} filas")

In [0]:
def exigir_columnas(tabla, columnas):
    faltan = [c for c in columnas if c not in spark.table(tabla).columns]
    assert not faltan, f"A {tabla} le faltan las columnas {faltan}. Vuelve a ejecutar el notebook de Plata correspondiente."

In [0]:
## Verificar que las tablas esten listas
exigir_columnas(s_viajes, [
    "viaje_id", "genero", "edad", "bici_id", "estacion_retiro", "estacion_arribo",
    "estacion_retiro_es_rango", "estacion_arribo_es_rango", "ts_retiro", "ts_arribo",
    "duracion_min", "fecha_retiro", "hora_clima",
])
exigir_columnas(s_estaciones, [
    "num_cicloe", "id_ubicacion", "calle_principal", "calle_secundaria", "colonia", "alcaldia",
    "latitud", "longitud", "tipo_sitio", "estatus",
])
exigir_columnas(s_clima, [
    "fecha_hora", "fecha", "hora", "temperatura_c", "humedad_pct", "precipitacion_mm",
    "viento_kmh", "descripcion_clima", "llueve",
])

In [0]:
## listas para dim_fecha
DIAS  = ["Domingo", "Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado"]   # dayofweek: 1 = domingo
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
         "Septiembre", "Octubre", "Noviembre", "Diciembre"]

In [0]:
limites = spark.table(s_viajes).agg(
    F.min("fecha_retiro").alias("primero"), F.max("fecha_retiro").alias("ultimo")
).first()
assert limites["primero"] is not None, "silver.viajes está vacío"

In [0]:
inicio  = limites["primero"].replace(day=1)
ultimo  = limites["ultimo"]
fin     = ultimo.replace(day=calendar.monthrange(ultimo.year, ultimo.month)[1])
fechas  = [(inicio + timedelta(days=i),) for i in range((fin - inicio).days + 1)]
print("Calendario:", inicio, "->", fin, f"({len(fechas)} días)")

Calendario: 2026-06-01 -> 2026-10-31 (153 días)


In [0]:
dias_arr  = F.array(*[F.lit(d) for d in DIAS])
meses_arr = F.array(*[F.lit(m) for m in MESES])

In [0]:
dim_fecha = (
    spark.createDataFrame(fechas, "fecha date")
    .select(
        F.date_format("fecha", "yyyyMMdd").cast("int").alias("fecha_key"),
        "fecha",
        F.year("fecha").alias("anio"),
        F.quarter("fecha").alias("trimestre"),
        F.month("fecha").alias("mes"),
        F.element_at(meses_arr, F.month("fecha")).alias("nombre_mes"),
        F.dayofmonth("fecha").alias("dia"),
        F.dayofweek("fecha").alias("dia_semana_num"),
        F.element_at(dias_arr, F.dayofweek("fecha")).alias("nombre_dia"),
        F.dayofweek("fecha").isin(1, 7).alias("es_fin_de_semana"),
    )
)
guardar(dim_fecha, G_DIM_FECHA)

catalog_ecobici_dev.golden.dim_fecha: 153 filas


In [0]:
dim_estacion = (
    spark.table(s_estaciones)
    .select(
        F.col("num_cicloe").alias("estacion_key"),
        "id_ubicacion",
        F.concat_ws(" y ", "calle_principal", "calle_secundaria").alias("nombre_estacion"),
        "calle_principal",
        "calle_secundaria",
        "colonia",
        "alcaldia",
        "latitud",
        "longitud",
        "tipo_sitio",
        "estatus",
    )
)
assert dim_estacion.select("estacion_key").distinct().count() == dim_estacion.count(), "estacion_key repetido"
guardar(dim_estacion, G_DIM_ESTACION)

catalog_ecobici_dev.golden.dim_estacion: 689 filas


In [0]:
GENEROS = [("M", "Masculino"), ("F", "Femenino"), ("O", "Otro"), ("ND", "No disponible")]
RANGOS_EDAD = [
    ("14-17", 14, 17), ("18-24", 18, 24), ("25-34", 25, 34), ("35-44", 35, 44),
    ("45-54", 45, 54), ("55-64", 55, 64), ("65+", 65, 200),
]

In [0]:
def rango_edad(edad):
    """Convierte la edad (columna) en su rango. Si no cae en ninguno, 'Sin dato'."""
    expresion = None
    for etiqueta, minimo, maximo in RANGOS_EDAD:
        condicion = edad.between(minimo, maximo)
        expresion = F.when(condicion, etiqueta) if expresion is None else expresion.when(condicion, etiqueta)
    return expresion.otherwise("Sin dato")

In [0]:
etiquetas_edad = [r[0] for r in RANGOS_EDAD] + ["Sin dato"]
filas = [
    (i + 1, genero, descripcion, rango)
    for i, ((genero, descripcion), rango) in enumerate(product(GENEROS, etiquetas_edad))
]
dim_usuario = spark.createDataFrame(
    filas, "usuario_key int, genero string, genero_desc string, rango_edad string"
)
guardar(dim_usuario, G_DIM_USUARIO)

catalog_ecobici_dev.golden.dim_usuario: 32 filas


In [0]:
clima = spark.table(s_clima)
 
rango_temperatura = (
    F.when(F.col("temperatura_c") < 12, "Frío (menos de 12°C)")
     .when(F.col("temperatura_c") < 18, "Fresco (12 a 18°C)")
     .when(F.col("temperatura_c") < 24, "Templado (18 a 24°C)")
     .otherwise("Cálido (24°C o más)")
)

In [0]:
clima_atributos = clima.select(
    "fecha_hora",
    F.date_format("fecha", "yyyyMMdd").cast("int").alias("fecha_key"),
    "hora",
    "descripcion_clima",
    "llueve",
    rango_temperatura.alias("rango_temperatura"),
    "temperatura_c",
    "humedad_pct",
    "precipitacion_mm",
    "viento_kmh",
)

In [0]:
combinaciones = clima_atributos.select("descripcion_clima", "llueve", "rango_temperatura").distinct()
orden = Window.orderBy("descripcion_clima", "llueve", "rango_temperatura")
 
sin_dato = spark.createDataFrame(
    [(-1, "Sin dato", False, "Sin dato")],
    "clima_key int, descripcion_clima string, llueve boolean, rango_temperatura string",
)

In [0]:
dim_clima = (
    combinaciones
    .select(
        F.row_number().over(orden).alias("clima_key"),
        "descripcion_clima", "llueve", "rango_temperatura",
    )
    .unionByName(sin_dato)
)
guardar(dim_clima, G_DIM_CLIMA)

catalog_ecobici_dev.golden.dim_clima: 31 filas


In [0]:
fact_clima_hora = (
    clima_atributos
    .join(
        spark.table(G_DIM_CLIMA).filter(F.col("clima_key") > 0),
        ["descripcion_clima", "llueve", "rango_temperatura"],
        "left",
    )
    .select(
        "fecha_hora", "fecha_key", "hora", "clima_key",
        "temperatura_c", "humedad_pct", "precipitacion_mm", "viento_kmh",
    )
)
assert fact_clima_hora.filter(F.col("clima_key").isNull()).count() == 0, "Hay horas de clima sin clima_key"
guardar(fact_clima_hora, G_FACT_CLIMA)

catalog_ecobici_dev.golden.fact_clima_hora: 2,952 filas


In [0]:
viajes = spark.table(s_viajes).withColumn("rango_edad", rango_edad(F.col("edad")))
 
usuarios = F.broadcast(spark.table(G_DIM_USUARIO).select("usuario_key", "genero", "rango_edad"))
clima_hora = F.broadcast(
    spark.table(G_FACT_CLIMA).select(
        F.col("fecha_hora").alias("hora_clima"), "clima_key", "temperatura_c", "precipitacion_mm"
    )
)
 
fact_viajes = (
    viajes
    .join(usuarios, ["genero", "rango_edad"], "left")
    .join(clima_hora, "hora_clima", "left")
    .select(
        "viaje_id",
        F.date_format("fecha_retiro", "yyyyMMdd").cast("int").alias("fecha_key"),
        F.hour("ts_retiro").alias("hora_retiro"),
        F.col("estacion_retiro").alias("estacion_retiro_key"),
        F.col("estacion_arribo").alias("estacion_arribo_key"),
        "usuario_key",
        F.coalesce("clima_key", F.lit(-1)).alias("clima_key"),
        "edad",
        "duracion_min",
        "ts_retiro",
        "ts_arribo",
        "bici_id",
        "estacion_retiro_es_rango",
        "estacion_arribo_es_rango",
        "temperatura_c",
        "precipitacion_mm",
    )
)
guardar(fact_viajes, G_FACT_VIAJES)

catalog_ecobici_dev.golden.fact_viajes: 6,065,074 filas


In [0]:
fact = spark.table(G_FACT_VIAJES)
n_fact   = fact.count()
n_silver = spark.table(s_viajes).count()
print(f"Viajes en Silver: {n_silver:,} | Viajes en fact_viajes: {n_fact:,}")
assert n_fact == n_silver, "fact_viajes no tiene los mismos viajes que silver.viajes"

Viajes en Silver: 6,065,074 | Viajes en fact_viajes: 6,065,074


In [0]:
assert fact.select("viaje_id").distinct().count() == n_fact, "Hay viaje_id repetidos"

In [0]:
def huerfanos(columna_hecho, tabla_dim, columna_dim):
    """Viajes cuya llave no existe en la dimensión (incluye llaves nulas)."""
    dim = spark.table(tabla_dim).select(F.col(columna_dim).alias("_llave"))
    return fact.join(dim, fact[columna_hecho] == F.col("_llave"), "left_anti").count()

In [0]:
for columna, tabla, llave in [
    ("fecha_key", G_DIM_FECHA, "fecha_key"),
    ("estacion_retiro_key", G_DIM_ESTACION, "estacion_key"),
    ("estacion_arribo_key", G_DIM_ESTACION, "estacion_key"),
    ("usuario_key", G_DIM_USUARIO, "usuario_key"),
    ("clima_key", G_DIM_CLIMA, "clima_key"),
]:
    n = huerfanos(columna, tabla, llave)
    print(f"Integridad {columna:22s} -> {tabla.split('.')[-1]:15s} huérfanos: {n}")
    assert n == 0, f"{columna} tiene {n} valores que no existen en la dimensión"
 
sin_clima = fact.filter(F.col("clima_key") == -1).count()
print(f"Viajes sin clima para su hora: {sin_clima:,} ({sin_clima / n_fact * 100:.3f}%)")

Integridad fecha_key              -> dim_fecha       huérfanos: 0
Integridad estacion_retiro_key    -> dim_estacion    huérfanos: 0
Integridad estacion_arribo_key    -> dim_estacion    huérfanos: 0
Integridad usuario_key            -> dim_usuario     huérfanos: 0
Integridad clima_key              -> dim_clima       huérfanos: 0
Viajes sin clima para su hora: 783 (0.013%)


In [0]:
display(fact.limit(20))

viaje_id,fecha_key,hora_retiro,estacion_retiro_key,estacion_arribo_key,usuario_key,clima_key,edad,duracion_min,ts_retiro,ts_arribo,bici_id,estacion_retiro_es_rango,estacion_arribo_es_rango,temperatura_c,precipitacion_mm
9da9cd3b4660dcc93be325b97f37abbfeadb8ac3059b5da9099ed8e176eb699c,20260704,12,584,575,11,30,25,6.75,2026-07-04T12:58:24Z,2026-07-04T13:05:09Z,2000461,false,false,21.3,0.0
9a2f462c39aecb6f2789a82722327e4c02208f2cda07ece51df146359a420f89,20260713,17,316,373,4,10,44,41.28333333333333,2026-07-13T17:06:43Z,2026-07-13T17:48:00Z,2000461,false,false,21.8,0.4
028219acd6655e1255140d8c917734d8c7a7c2d89eb2958e0b2e484c4249497b,20260713,18,401,392,4,30,38,10.633333333333333,2026-07-13T18:04:59Z,2026-07-13T18:15:37Z,2000461,false,false,18.2,0.0
55ccb087fc4003c6dd65c91ee6c6727571b96eb38d2d777782c2d7197717cdba,20260713,19,372,369,11,10,25,6.25,2026-07-13T19:32:19Z,2026-07-13T19:38:34Z,2000461,false,false,18.4,0.1
aeeefb6fa3531207864d182fc1d66b84e176fa49724298a121f2ec3e8c62e85d,20260713,21,261,86,5,20,50,10.05,2026-07-13T21:28:46Z,2026-07-13T21:38:49Z,2000461,false,false,16.6,0.0
35f007d02e50d9f7b8cc7874e0df1240ebd3ff8878e580dba773b63781140dad,20260714,16,91,563,6,23,63,30.933333333333334,2026-07-14T16:05:29Z,2026-07-14T16:36:25Z,2000461,false,false,24.2,0.0
8654cc3ce3d403cff5080276c9360a1c130c3b7212b911c3f6df3f67732992fd,20260715,7,495,270,3,8,30,11.416666666666666,2026-07-15T07:31:58Z,2026-07-15T07:43:23Z,2000461,false,false,14.5,0.1
976fc4471923d25dd689ee33c2ceaec7a5729319da75e859cff1559ee3be2534,20260715,9,158,51,5,8,46,13.45,2026-07-15T09:15:48Z,2026-07-15T09:29:15Z,2000461,true,false,17.2,0.2
9f3330c4eaaa627406faaaf007181c71c23dd8edae05339af44153613826b10d,20260715,13,27,27,4,10,35,32.6,2026-07-15T13:16:07Z,2026-07-15T13:48:43Z,2000461,false,false,23.5,0.2
50d17d9dc34353719af2b2ac1b13f1a05ca232d6f68612ffea1d8feff85a7986,20260715,16,65,266,5,13,48,10.833333333333334,2026-07-15T16:21:13Z,2026-07-15T16:32:03Z,2000461,false,true,22.2,0.8


In [0]:
display(spark.table(G_DIM_CLIMA).orderBy("clima_key"))

clima_key,descripcion_clima,llueve,rango_temperatura
-1,Sin dato,false,Sin dato
1,Despejado,false,Cálido (24°C o más)
2,Despejado,false,Fresco (12 a 18°C)
3,Despejado,false,Frío (menos de 12°C)
4,Despejado,false,Templado (18 a 24°C)
5,Llovizna intensa,true,Fresco (12 a 18°C)
6,Llovizna intensa,true,Templado (18 a 24°C)
7,Llovizna ligera,true,Cálido (24°C o más)
8,Llovizna ligera,true,Fresco (12 a 18°C)
9,Llovizna ligera,true,Frío (menos de 12°C)
